# LeetCode #176 — Second Highest Salary: SQL với `DISTINCT`, `ORDER BY`, `LIMIT/OFFSET` và `NULL`

**Mức độ:** Medium · **Chủ đề:** Database (SQL), Pandas

Đây là bài **cơ sở dữ liệu**: trên LeetCode bạn nộp một câu **SQL** (MySQL) hoặc một hàm **Pandas**. Notebook chạy SQL thật bằng `sqlite3` (có sẵn trong Python) và viết bản Pandas tương ứng.

Skeleton trên LeetCode:

```sql
-- MySQL
# Write your MySQL query statement below
```

```python
# Pandas
import pandas as pd

def second_highest_salary(employee: pd.DataFrame) -> pd.DataFrame:
```

Mục tiêu của notebook:

1. Đọc đề dạng bảng: cột, khóa chính, kết quả mong đợi (tên cột, khi nào là `null`).
2. `SELECT DISTINCT … ORDER BY … DESC LIMIT 1 OFFSET 1`.
3. Mẹo "bọc trong truy vấn con" để trả `NULL` khi không có dòng nào.
4. Cách khác: `MAX` của các lương **nhỏ hơn** `MAX`. Bản Pandas.

# 1. Bài toán là gì?

Bảng `Employee`:

```text
+-------------+------+
| Column Name | Type |
+-------------+------+
| id          | int  |
| salary      | int  |
+-------------+------+
id là khóa chính (mỗi dòng một giá trị id khác nhau).
Mỗi dòng là lương của một nhân viên.
```

Viết truy vấn tìm mức lương **cao thứ hai khác nhau** (distinct). Nếu không có (ví dụ mọi người cùng lương, hoặc chỉ có một người) thì trả `null`. Cột kết quả tên `SecondHighestSalary`.

---

## Ví dụ

```text
Employee                 Kết quả
+----+--------+          +---------------------+
| id | salary |          | SecondHighestSalary |
+----+--------+          +---------------------+
| 1  | 100    |          | 200                 |
| 2  | 200    |          +---------------------+
| 3  | 300    |
+----+--------+

Employee = [(1, 100)]  →  SecondHighestSalary = null
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. SQL cơ bản — truy vấn bảng dữ liệu
2. Pandas DataFrame — SQL bằng Python

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 SQL cơ bản — truy vấn bảng dữ liệu

### Bảng, hàng, cột

Cơ sở dữ liệu quan hệ lưu dữ liệu trong các **bảng**. Mỗi **hàng** là một bản ghi, mỗi **cột** là một thuộc tính có kiểu (số, chuỗi…). Ô có thể **rỗng** — giá trị đặc biệt `NULL` ("không biết / không có").

```text
Employee
+----+-------+--------+--------------+
| id | name  | salary | departmentId |
+----+-------+--------+--------------+
| 1  | Joe   | 70000  | 1            |
| 2  | Jim   | 90000  | 1            |
| 3  | Henry | 80000  | 2            |
+----+-------+--------+--------------+
```

Trong notebook ta chạy SQL thật bằng **`sqlite3`** (có sẵn trong Python, chạy trong bộ nhớ). LeetCode dùng MySQL — cú pháp các câu cơ bản giống nhau, chỗ khác biệt sẽ được ghi chú.

In [1]:
import sqlite3

def sql_demo(query, params=()):
    con = sqlite3.connect(":memory:")
    con.execute("CREATE TABLE Employee (id INTEGER, name TEXT, salary INTEGER, departmentId INTEGER)")
    con.executemany("INSERT INTO Employee VALUES (?, ?, ?, ?)",
                    [(1, "Joe", 70000, 1), (2, "Jim", 90000, 1), (3, "Henry", 80000, 2),
                     (4, "Sam", 60000, 2), (5, "Max", 90000, 1)])
    con.execute("CREATE TABLE Department (id INTEGER, name TEXT)")
    con.executemany("INSERT INTO Department VALUES (?, ?)", [(1, "IT"), (2, "Sales")])
    cur = con.execute(query, params)
    cols = [d[0] for d in cur.description]
    out = cur.fetchall()
    con.close()
    print(cols)
    for r in out:
        print("  ", r)

sql_demo("SELECT * FROM Employee")

['id', 'name', 'salary', 'departmentId']
   (1, 'Joe', 70000, 1)
   (2, 'Jim', 90000, 1)
   (3, 'Henry', 80000, 2)
   (4, 'Sam', 60000, 2)
   (5, 'Max', 90000, 1)


### Thứ tự **viết** và thứ tự **thực hiện** của một câu SELECT

```text
Viết:        SELECT ... FROM ... WHERE ... GROUP BY ... HAVING ... ORDER BY ... LIMIT ...
Thực hiện:   FROM → WHERE → GROUP BY → HAVING → SELECT → ORDER BY → LIMIT
```

Hiểu thứ tự thực hiện giải thích nhiều lỗi: `WHERE` chạy **trước** khi gom nhóm nên không dùng được hàm gộp (`COUNT`, `MAX`…) trong `WHERE` — phải dùng `HAVING`.

### Lọc, sắp xếp, giới hạn, bỏ trùng

In [2]:
print("Lương > 65000, giảm dần, lấy 2 hàng:")
sql_demo("SELECT name, salary FROM Employee WHERE salary > 65000 ORDER BY salary DESC LIMIT 2")
print("Các mức lương KHÁC NHAU, giảm dần, bỏ qua 1 hàng đầu (OFFSET):")
sql_demo("SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1")

Lương > 65000, giảm dần, lấy 2 hàng:
['name', 'salary']
   ('Jim', 90000)
   ('Max', 90000)
Các mức lương KHÁC NHAU, giảm dần, bỏ qua 1 hàng đầu (OFFSET):
['salary']
   (80000,)


### Hàm gộp và GROUP BY

`COUNT`, `SUM`, `AVG`, `MIN`, `MAX` gộp nhiều hàng thành một giá trị. `GROUP BY cột` chia bảng thành nhóm theo cột đó, gộp riêng từng nhóm. `HAVING` lọc **sau** khi gộp.

In [3]:
sql_demo("""
SELECT departmentId, COUNT(*) AS n, MAX(salary) AS top
FROM Employee
GROUP BY departmentId
HAVING COUNT(*) >= 2
""")

['departmentId', 'n', 'top']
   (1, 3, 90000)
   (2, 2, 80000)


### NULL: "không có giá trị"

- Hàm gộp trên **tập rỗng** (trừ `COUNT`) trả về `NULL` — đây là mẹo để trả về `NULL` khi "không có đáp án".
- So sánh với `NULL` bằng `=` luôn **không đúng** → phải dùng `IS NULL`.
- `IFNULL(x, d)` / `COALESCE(x, d)` thay `NULL` bằng giá trị mặc định.

In [4]:
sql_demo("SELECT MAX(salary) AS m FROM Employee WHERE salary > 1000000")
sql_demo("SELECT NULL = NULL AS bang_nhau, NULL IS NULL AS la_null, COALESCE(NULL, 0) AS mac_dinh")

['m']
   (None,)
['bang_nhau', 'la_null', 'mac_dinh']
   (None, 1, 0)


### Truy vấn con (subquery)

Một câu `SELECT` có thể nằm **bên trong** câu khác: trả về một giá trị (dùng trong `WHERE x > (...)`), một danh sách (dùng với `IN`), hoặc một bảng tạm (đặt trong `FROM`).

In [5]:
print("Lương cao thứ hai = lương lớn nhất trong số những lương nhỏ hơn lương lớn nhất:")
sql_demo("SELECT MAX(salary) AS second FROM Employee WHERE salary < (SELECT MAX(salary) FROM Employee)")

Lương cao thứ hai = lương lớn nhất trong số những lương nhỏ hơn lương lớn nhất:
['second']
   (80000,)


### JOIN: ghép hai bảng

`A JOIN B ON điều_kiện` tạo các cặp hàng (a, b) thỏa điều kiện.

- **INNER JOIN** (mặc định): chỉ giữ cặp khớp.
- **LEFT JOIN**: giữ **mọi** hàng của A; hàng không khớp thì cột của B là `NULL`.
- **Self-join**: ghép bảng với **chính nó** (đặt hai bí danh khác nhau) — để so sánh các hàng trong cùng bảng.

In [6]:
sql_demo("""
SELECT d.name AS department, e.name AS employee, e.salary
FROM Employee e JOIN Department d ON e.departmentId = d.id
ORDER BY d.name, e.salary DESC
""")
print("Self-join: cặp nhân viên cùng phòng, người trước lương thấp hơn người sau")
sql_demo("""
SELECT a.name, b.name FROM Employee a JOIN Employee b
ON a.departmentId = b.departmentId AND a.salary < b.salary
""")

['department', 'employee', 'salary']
   ('IT', 'Jim', 90000)
   ('IT', 'Max', 90000)
   ('IT', 'Joe', 70000)
   ('Sales', 'Henry', 80000)
   ('Sales', 'Sam', 60000)
Self-join: cặp nhân viên cùng phòng, người trước lương thấp hơn người sau
['name', 'name']
   ('Joe', 'Jim')
   ('Joe', 'Max')
   ('Sam', 'Henry')


### Lỗi hay gặp

- Quên `DISTINCT` khi hỏi "giá trị lớn thứ k **khác nhau**".
- Dùng hàm gộp trong `WHERE` (phải dùng `HAVING` hoặc truy vấn con).
- `LIMIT 1 OFFSET k` trên bảng **không đủ** hàng trả về **không hàng nào** (không phải `NULL`) — bọc trong truy vấn con hoặc dùng `MAX` để có `NULL`.
- So sánh `= NULL` thay vì `IS NULL`.
- Trong MySQL, `LIMIT` không nhận biểu thức (`LIMIT N-1` lỗi) → phải tính trước (`SET N = N - 1`).

## 2.2 Pandas DataFrame — SQL bằng Python

### DataFrame là gì?

`pandas.DataFrame` là một **bảng** trong bộ nhớ: các cột có tên, mỗi cột là một `Series` (mảng có nhãn). LeetCode có phiên bản Pandas cho các bài SQL; mỗi thao tác SQL có một tương đương:

| SQL | Pandas |
|---|---|
| `WHERE cond` | `df[df["col"] > x]` |
| `SELECT a, b` | `df[["a", "b"]]` |
| `ORDER BY a DESC` | `df.sort_values("a", ascending=False)` |
| `DISTINCT` | `df.drop_duplicates()` / `series.unique()` |
| `GROUP BY` + gộp | `df.groupby("k")["v"].max()` |
| `JOIN` | `df1.merge(df2, left_on=..., right_on=...)` |
| `DENSE_RANK()` | `series.rank(method="dense", ascending=False)` |
| `LAG(col)` | `series.shift(1)` |

In [7]:
import pandas as pd

emp = pd.DataFrame({"id": [1, 2, 3, 4, 5],
                    "name": ["Joe", "Jim", "Henry", "Sam", "Max"],
                    "salary": [70000, 90000, 80000, 60000, 90000],
                    "departmentId": [1, 1, 2, 2, 1]})
dep = pd.DataFrame({"id": [1, 2], "name": ["IT", "Sales"]})
print(emp)

   id   name  salary  departmentId
0   1    Joe   70000             1
1   2    Jim   90000             1
2   3  Henry   80000             2
3   4    Sam   60000             2
4   5    Max   90000             1


### Lọc, chọn cột, sắp xếp, bỏ trùng

In [8]:
print(emp[emp["salary"] > 65000][["name", "salary"]].sort_values("salary", ascending=False).head(2))
print("lương khác nhau giảm dần:", sorted(emp["salary"].unique().tolist(), reverse=True))

  name  salary
1  Jim   90000
4  Max   90000
lương khác nhau giảm dần: [90000, 80000, 70000, 60000]


### groupby và transform

- `groupby(k)[v].max()` → **co** mỗi nhóm thành một giá trị (như `GROUP BY`).
- `groupby(k)[v].transform("max")` → trả về giá trị của nhóm **cho từng hàng** (giống hàm cửa sổ `MAX() OVER (PARTITION BY k)`), rất tiện để lọc "hàng có giá trị lớn nhất nhóm".

In [9]:
print(emp.groupby("departmentId")["salary"].max())
top = emp[emp["salary"] == emp.groupby("departmentId")["salary"].transform("max")]
print(top)

departmentId
1    90000
2    80000
Name: salary, dtype: int64
   id   name  salary  departmentId
1   2    Jim   90000             1
2   3  Henry   80000             2
4   5    Max   90000             1


### merge (JOIN), rank, shift

In [10]:
m = emp.merge(dep, left_on="departmentId", right_on="id", suffixes=("", "_dep"))
print(m[["name_dep", "name", "salary"]])
s = pd.Series([4.00, 4.00, 3.85, 3.65])
print("rank dense:", s.rank(method="dense", ascending=False).tolist(), "| rank min (như RANK):", s.rank(method="min", ascending=False).tolist())
print("shift(1) (như LAG):", s.shift(1).tolist())

  name_dep   name  salary
0       IT    Joe   70000
1       IT    Jim   90000
2    Sales  Henry   80000
3    Sales    Sam   60000
4       IT    Max   90000
rank dense: [1.0, 1.0, 2.0, 3.0] | rank min (như RANK): [1.0, 1.0, 3.0, 4.0]
shift(1) (như LAG): [nan, 4.0, 4.0, 3.85]


### Lỗi hay gặp

- `df[...]` lọc trả về DataFrame **có thể rỗng** — kiểm tra `len(...)` trước khi lấy `.iloc[0]`.
- `rank` trả về số thực (`1.0`) → `astype(int)` nếu cần.
- `merge` hai bảng cùng tên cột → cột bị thêm hậu tố `_x`, `_y` (hoặc `suffixes` tự đặt).
- Khi đề yêu cầu trả `null`, trả `None` trong một DataFrame một hàng — không trả DataFrame rỗng.

## 2.3 Áp dụng vào bài này

Lương cao thứ hai **khác nhau** = `DISTINCT` + `ORDER BY salary DESC` + `LIMIT 1 OFFSET 1`. Cái bẫy là khi không có lương thứ hai phải trả **`NULL`** chứ không phải bảng rỗng — bọc truy vấn trong một truy vấn con `SELECT (...)`, hoặc dùng `MAX(salary) WHERE salary < (SELECT MAX(...))` vì hàm gộp trên tập rỗng cho `NULL`. Phiên bản Pandas dùng `drop_duplicates`/`unique`, sắp xếp và kiểm tra độ dài trước khi lấy.

# 3. Nền tảng: chạy SQL trong notebook

`sqlite3` là cơ sở dữ liệu nhỏ có sẵn trong Python. Ta đưa bảng (dưới dạng `DataFrame` của Pandas) vào một CSDL **trong bộ nhớ**, chạy câu SQL, nhận lại `DataFrame`. Phần lớn cú pháp trong bài này giống hệt MySQL.

In [11]:
import sqlite3
import random
import pandas as pd


def run_sql(sql: str, tables: dict) -> pd.DataFrame:
    """tables = {"TênBảng": DataFrame}. Tạo CSDL mới trong bộ nhớ, chạy sql, trả DataFrame."""
    conn = sqlite3.connect(":memory:")
    for name, df in tables.items():
        df.to_sql(name, conn, index=False)
    out = pd.read_sql_query(sql, conn)
    conn.close()
    return out


def rows(df: pd.DataFrame) -> list:
    """DataFrame → list các dòng; NULL/NaN đổi thành None để dễ so sánh."""
    return [[None if pd.isna(v) else v for v in r] for r in df.itertuples(index=False)]


def emp_df(salaries) -> pd.DataFrame:
    return pd.DataFrame({"id": range(1, len(salaries) + 1), "salary": salaries})


employee = emp_df([100, 200, 300])
print(employee)
print(run_sql("SELECT * FROM Employee WHERE salary > 150", {"Employee": employee}))

   id  salary
0   1     100
1   2     200
2   3     300
   id  salary
0   2     200
1   3     300


# 4. Từng bước xây truy vấn

## Bước 1 — Lấy các mức lương **khác nhau**, sắp giảm dần

`DISTINCT` bỏ các dòng trùng; `ORDER BY salary DESC` sắp giảm dần.

In [12]:
emp2 = emp_df([300, 100, 300, 200, 200])
print(run_sql("SELECT DISTINCT salary FROM Employee ORDER BY salary DESC", {"Employee": emp2}))

   salary
0     300
1     200
2     100


## Bước 2 — Bỏ dòng đầu, lấy một dòng: `LIMIT 1 OFFSET 1`

`OFFSET k` bỏ `k` dòng đầu; `LIMIT m` chỉ lấy tối đa `m` dòng. (MySQL còn viết `LIMIT 1, 1` = "bỏ 1, lấy 1".)

In [13]:
q_step2 = """
SELECT DISTINCT salary AS SecondHighestSalary
FROM Employee
ORDER BY salary DESC
LIMIT 1 OFFSET 1
"""
print(run_sql(q_step2, {"Employee": emp2}))
print("--- chỉ một nhân viên:")
print(run_sql(q_step2, {"Employee": emp_df([100])}))

   SecondHighestSalary
0                  200
--- chỉ một nhân viên:
Empty DataFrame
Columns: [SecondHighestSalary]
Index: []


Với một nhân viên, kết quả là **bảng rỗng** (0 dòng) — nhưng đề yêu cầu **một dòng chứa `null`**.

## Bước 3 — Bọc vào truy vấn con vô hướng để có `NULL`

Một **truy vấn con vô hướng** (scalar subquery) đặt trong `SELECT (…)` trả về **một giá trị**; nếu truy vấn con không có dòng nào, giá trị đó là `NULL`. Truy vấn ngoài không có `FROM` nên luôn trả đúng **một dòng**.

In [14]:
q_final = """
SELECT (
    SELECT DISTINCT salary
    FROM Employee
    ORDER BY salary DESC
    LIMIT 1 OFFSET 1
) AS SecondHighestSalary
"""
for sal in [[100, 200, 300], [100], [100, 100], [300, 100, 300, 200, 200]]:
    print(f"{str(sal):<26} →", rows(run_sql(q_final, {"Employee": emp_df(sal)})))

[100, 200, 300]            → [[200]]
[100]                      → [[None]]
[100, 100]                 → [[None]]
[300, 100, 300, 200, 200]  → [[200]]


(Trong MySQL có thể viết thêm `IFNULL((…), NULL)` cho rõ ý; không bắt buộc.)

# 5. Cách thứ hai: `MAX` của các lương nhỏ hơn lương lớn nhất

Lương cao thứ hai (khác nhau) = lương **lớn nhất** trong số các lương **nhỏ hơn** lương lớn nhất. `MAX` trên tập rỗng trả `NULL` — tự xử lý trường hợp không có.

```sql
SELECT MAX(salary) AS SecondHighestSalary
FROM Employee
WHERE salary < (SELECT MAX(salary) FROM Employee)
```

In [15]:
q_max = """
SELECT MAX(salary) AS SecondHighestSalary
FROM Employee
WHERE salary < (SELECT MAX(salary) FROM Employee)
"""
for sal in [[100, 200, 300], [100], [100, 100], [300, 100, 300, 200, 200]]:
    print(f"{str(sal):<26} →", rows(run_sql(q_max, {"Employee": emp_df(sal)})))

[100, 200, 300]            → [[200]]
[100]                      → [[None]]
[100, 100]                 → [[None]]
[300, 100, 300, 200, 200]  → [[200]]


Cách này chỉ dùng được cho "thứ **hai**"; muốn thứ `N` tổng quát thì dùng `LIMIT/OFFSET` hoặc hàm cửa sổ (bài #177, #178).

# 6. Mô phỏng: truy vấn chạy theo thứ tự nào?

SQL được **viết** theo thứ tự `SELECT … FROM … WHERE … ORDER BY … LIMIT`, nhưng được **thực hiện** theo thứ tự logic:

```text
FROM  →  WHERE  →  GROUP BY  →  HAVING  →  SELECT (và DISTINCT)  →  ORDER BY  →  LIMIT/OFFSET
```

In [16]:
def trace_query(salaries):
    print("FROM Employee            :", salaries)
    distinct = sorted(set(salaries), key=lambda x: -x)
    print("SELECT DISTINCT + ORDER  :", distinct)
    after = distinct[1:2]
    print("LIMIT 1 OFFSET 1         :", after, "(rỗng)" if not after else "")
    val = after[0] if after else None
    print("bọc truy vấn con vô hướng →", val)
    return val

_ = trace_query([300, 100, 300, 200, 200])
_ = trace_query([500, 500])

FROM Employee            : [300, 100, 300, 200, 200]
SELECT DISTINCT + ORDER  : [300, 200, 100]
LIMIT 1 OFFSET 1         : [200] 
bọc truy vấn con vô hướng → 200
FROM Employee            : [500, 500]
SELECT DISTINCT + ORDER  : [500]
LIMIT 1 OFFSET 1         : [] (rỗng)
bọc truy vấn con vô hướng → None


# 7. Phiên bản nộp LeetCode

**MySQL:**

```sql
SELECT (
    SELECT DISTINCT salary
    FROM Employee
    ORDER BY salary DESC
    LIMIT 1 OFFSET 1
) AS SecondHighestSalary;
```

**Pandas:**

In [17]:
def second_highest_salary(employee: pd.DataFrame) -> pd.DataFrame:
    distinct = employee["salary"].drop_duplicates().sort_values(ascending=False)
    value = distinct.iloc[1] if len(distinct) >= 2 else None
    return pd.DataFrame({"SecondHighestSalary": [value]})

print(second_highest_salary(emp_df([100, 200, 300])))
print(second_highest_salary(emp_df([100])))

   SecondHighestSalary
0                  200
  SecondHighestSalary
0                None


In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<18} PASS  {actual}")


def expected_second(salaries):
    d = sorted(set(salaries), reverse=True)
    return [[d[1]]] if len(d) >= 2 else [[None]]


SQL_SOLUTION = q_final
check("Ví dụ 1 (SQL)", rows(run_sql(SQL_SOLUTION, {"Employee": emp_df([100, 200, 300])})), [[200]])
check("Ví dụ 2 (SQL)", rows(run_sql(SQL_SOLUTION, {"Employee": emp_df([100])})), [[None]])
check("Cùng lương (SQL)", rows(run_sql(SQL_SOLUTION, {"Employee": emp_df([7, 7, 7])})), [[None]])
check("Ví dụ 1 (Pandas)", rows(second_highest_salary(emp_df([100, 200, 300]))), [[200]])
check("Ví dụ 2 (Pandas)", rows(second_highest_salary(emp_df([100]))), [[None]])
check("Tên cột", list(run_sql(SQL_SOLUTION, {"Employee": emp_df([1, 2])}).columns), ["SecondHighestSalary"])

random.seed(176)
for _ in range(500):
    sal = [random.randint(1, 6) * 100 for _ in range(random.randint(1, 8))]
    exp = expected_second(sal)
    t = {"Employee": emp_df(sal)}
    assert rows(run_sql(q_final, t)) == exp == rows(run_sql(q_max, t)) == rows(second_highest_salary(emp_df(sal))), sal
print("500 bảng ngẫu nhiên: hai câu SQL và bản Pandas khớp đáp án tính bằng Python")

Ví dụ 1 (SQL)      PASS  [[200]]
Ví dụ 2 (SQL)      PASS  [[None]]
Cùng lương (SQL)   PASS  [[None]]
Ví dụ 1 (Pandas)   PASS  [[200]]
Ví dụ 2 (Pandas)   PASS  [[None]]
Tên cột            PASS  ['SecondHighestSalary']


500 bảng ngẫu nhiên: hai câu SQL và bản Pandas khớp đáp án tính bằng Python


# 8. Vì sao đúng? Độ phức tạp?

- `DISTINCT` + `ORDER BY salary DESC` tạo dãy các mức lương khác nhau giảm dần; phần tử thứ hai của dãy (bỏ 1, lấy 1) đúng là "cao thứ hai khác nhau".
- Truy vấn con vô hướng biến "không có dòng" thành `NULL`, truy vấn ngoài không có `FROM` nên luôn có đúng một dòng.

Độ phức tạp do CSDL quyết định: thường `O(n log n)` cho sắp xếp (hoặc `O(n)` nếu có chỉ mục trên `salary`). Cách `MAX` cần hai lượt quét `O(n)`.

# 9. Những lỗi dễ mắc

## Lỗi 1: Quên `DISTINCT`

In [19]:
q_no_distinct = """
SELECT (SELECT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1) AS SecondHighestSalary
"""
t1 = {"Employee": emp_df([300, 300, 200])}
print("Sai :", rows(run_sql(q_no_distinct, t1)), "| Đúng:", rows(run_sql(q_final, t1)))

Sai : [[300]] | Đúng: [[200]]


Hai người cùng lương 300 → dòng thứ hai vẫn là 300. Đề hỏi mức lương **khác nhau**.

## Lỗi 2: Không bọc truy vấn con → trả bảng rỗng thay vì `null`

In [20]:
t2 = {"Employee": emp_df([100])}
print("Sai : số dòng =", len(run_sql(q_step2, t2)), "| Đúng:", rows(run_sql(q_final, t2)))

Sai : số dòng = 0 | Đúng: [[None]]


## Lỗi 3: Dùng `ORDER BY salary` (tăng dần)

Tăng dần thì `OFFSET 1` cho lương **thấp thứ hai**. Luôn kiểm tra chiều sắp xếp `DESC`.

In [21]:
q_asc = """
SELECT (SELECT DISTINCT salary FROM Employee ORDER BY salary LIMIT 1 OFFSET 1) AS SecondHighestSalary
"""
t3 = {"Employee": emp_df([100, 200, 300, 400])}
print("Sai :", rows(run_sql(q_asc, t3)), "| Đúng:", rows(run_sql(q_final, t3)))

Sai : [[200]] | Đúng: [[300]]


## Lỗi 4: Đặt tên cột sai

LeetCode so sánh cả **tên cột**. `SELECT … AS second_highest` sẽ bị chấm sai dù giá trị đúng.

# 10. Mẹo nhận diện pattern

- **"Giá trị lớn thứ N"** → `DISTINCT` + `ORDER BY … DESC` + `LIMIT 1 OFFSET N−1`, hoặc `DENSE_RANK()`.
- **"Không có thì trả null"** → bọc trong truy vấn con vô hướng, hoặc dùng hàm tổng hợp (`MAX`, `MIN` trả `NULL` trên tập rỗng).
- **Đọc kỹ tên cột kết quả** — một phần của đáp án.

Bài liên quan: **#177 Nth Highest Salary**, **#178 Rank Scores**, **#184 Department Highest Salary**, **#185 Department Top Three Salaries** (Hard).

# 11. Bài tập vận dụng

Mỗi bài viết một hàm **trả về chuỗi SQL** (hoặc một hàm Pandas). Các `assert` chạy câu SQL của bạn trên `sqlite3`.

## Bài 1 — Tính bằng tay

Với lương `[400, 100, 400, 300, 300, 200]`, ghi kết quả từng bước: `DISTINCT + ORDER BY`, `LIMIT 1 OFFSET 1`.

## Bài 2 — Viết lại câu SQL của bài gốc

In [22]:
def exercise_2_sql() -> str:
    # TODO: trả về chuỗi SQL
    pass


# assert rows(run_sql(exercise_2_sql(), {"Employee": emp_df([100, 200, 300])})) == [[200]]
# assert rows(run_sql(exercise_2_sql(), {"Employee": emp_df([100])})) == [[None]]

## Bài 3 — Biến thể: lương cao thứ **ba** khác nhau (cột `ThirdHighestSalary`)

In [23]:
def exercise_3_sql() -> str:
    # TODO
    pass


# assert rows(run_sql(exercise_3_sql(), {"Employee": emp_df([100, 200, 300, 300])})) == [[100]]
# assert rows(run_sql(exercise_3_sql(), {"Employee": emp_df([100, 200])})) == [[None]]

## Bài 4 — Biến thể: lương **thấp** thứ hai khác nhau, dùng cách `MIN` (cột `SecondLowestSalary`)

In [24]:
def exercise_4_sql() -> str:
    # TODO
    pass


# assert rows(run_sql(exercise_4_sql(), {"Employee": emp_df([300, 100, 200, 100])})) == [[200]]
# assert rows(run_sql(exercise_4_sql(), {"Employee": emp_df([5, 5])})) == [[None]]

## Bài 5 — Mini challenge: bản Pandas tổng quát cho "lương cao thứ `k`"

Trả về `DataFrame` một dòng, cột `f"Top{k}Salary"`, giá trị `None` nếu không đủ `k` mức lương khác nhau.

In [25]:
def exercise_5_kth_salary(employee: pd.DataFrame, k: int) -> pd.DataFrame:
    # TODO
    pass


# assert rows(exercise_5_kth_salary(emp_df([100, 200, 300]), 1)) == [[300]]
# assert rows(exercise_5_kth_salary(emp_df([100, 200, 200]), 2)) == [[100]]
# assert rows(exercise_5_kth_salary(emp_df([100]), 2)) == [[None]] and list(exercise_5_kth_salary(emp_df([100]), 2).columns) == ["Top2Salary"]

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_query([400, 100, 400, 300, 300, 200])` sau khi tự làm.

**Bài 3**
- Mức 1: `OFFSET 2`.

**Bài 4**
- Mức 1: `MIN(salary)` với điều kiện `salary > (SELECT MIN(salary) FROM Employee)`.

**Bài 5**
- Mức 1: `drop_duplicates().sort_values(ascending=False)`, rồi `iloc[k - 1]` nếu đủ dài.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [26]:
_ = trace_query([400, 100, 400, 300, 300, 200])

FROM Employee            : [400, 100, 400, 300, 300, 200]
SELECT DISTINCT + ORDER  : [400, 300, 200, 100]
LIMIT 1 OFFSET 1         : [300] 
bọc truy vấn con vô hướng → 300


## Đáp án Bài 2

In [27]:
def answer_2_sql() -> str:
    return """
    SELECT (
        SELECT DISTINCT salary FROM Employee
        ORDER BY salary DESC
        LIMIT 1 OFFSET 1
    ) AS SecondHighestSalary
    """

assert rows(run_sql(answer_2_sql(), {"Employee": emp_df([100, 200, 300])})) == [[200]]
assert rows(run_sql(answer_2_sql(), {"Employee": emp_df([100])})) == [[None]]
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [28]:
def answer_3_sql() -> str:
    return """
    SELECT (
        SELECT DISTINCT salary FROM Employee
        ORDER BY salary DESC
        LIMIT 1 OFFSET 2
    ) AS ThirdHighestSalary
    """

assert rows(run_sql(answer_3_sql(), {"Employee": emp_df([100, 200, 300, 300])})) == [[100]]
assert rows(run_sql(answer_3_sql(), {"Employee": emp_df([100, 200])})) == [[None]]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [29]:
def answer_4_sql() -> str:
    return """
    SELECT MIN(salary) AS SecondLowestSalary
    FROM Employee
    WHERE salary > (SELECT MIN(salary) FROM Employee)
    """

assert rows(run_sql(answer_4_sql(), {"Employee": emp_df([300, 100, 200, 100])})) == [[200]]
assert rows(run_sql(answer_4_sql(), {"Employee": emp_df([5, 5])})) == [[None]]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [30]:
def answer_5_kth_salary(employee: pd.DataFrame, k: int) -> pd.DataFrame:
    distinct = employee["salary"].drop_duplicates().sort_values(ascending=False)
    value = distinct.iloc[k - 1] if len(distinct) >= k else None
    return pd.DataFrame({f"Top{k}Salary": [value]})

assert rows(answer_5_kth_salary(emp_df([100, 200, 300]), 1)) == [[300]]
assert rows(answer_5_kth_salary(emp_df([100, 200, 200]), 2)) == [[100]]
assert rows(answer_5_kth_salary(emp_df([100]), 2)) == [[None]] and list(answer_5_kth_salary(emp_df([100]), 2).columns) == ["Top2Salary"]
for _ in range(300):
    sal = [random.randint(1, 5) for _ in range(random.randint(1, 7))]
    k = random.randint(1, 4)
    d = sorted(set(sal), reverse=True)
    assert rows(answer_5_kth_salary(emp_df(sal), k)) == [[d[k - 1] if len(d) >= k else None]]
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> `SELECT (SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1) AS SecondHighestSalary` — `DISTINCT` để bỏ lương trùng, `DESC` để lấy từ cao xuống, `OFFSET 1` để bỏ mức cao nhất, truy vấn con vô hướng để có `NULL` khi không tồn tại. Hoặc `MAX(salary) WHERE salary < MAX`.

---

## Checklist trước khi nộp

- [ ] Có `DISTINCT`?
- [ ] Sắp `DESC`?
- [ ] Trả **một dòng `null`** khi không có (không phải bảng rỗng)?
- [ ] Tên cột đúng `SecondHighestSalary`?